# Notebook 4 - Evaluation and Efficiency Metrics

Computes the metrics from spec section 7: category/severity distribution, auto-route vs human-review rate, classification accuracy vs gold labels, and a latency/throughput snapshot.

**Throughput note:** the per-complaint flow is logically serial (two LLM calls each). For meaningful throughput on the GPU, batch the same stage across many complaints into grouped vLLM requests; the numbers below reflect serial mock execution.

In [ ]:
import json, time
from pathlib import Path
import pandas as pd
from complaint_router.pipeline import run_pipeline
from complaint_router.llm.mock_client import MockClient
from complaint_router.schemas import ComplaintInput

rows = json.loads((Path('..') / 'data' / 'sample_complaints.json').read_text(encoding='utf-8'))
client = MockClient()

results, t0 = [], time.perf_counter()
for r in rows:
    c = ComplaintInput(**{k: r[k] for k in ('complaint_id', 'channel', 'complaint_text', 'received_at')})
    res = run_pipeline(c, client)
    results.append((r, res))
elapsed = time.perf_counter() - t0

## Category and severity distribution

In [ ]:
cats = pd.Series([res.classification.category.value for _, res in results if res.classification])
sevs = pd.Series([res.analysis.severity_level.value for _, res in results if res.analysis])
print(cats.value_counts())
print(sevs.value_counts())

## Auto-route vs human-review rate

In [ ]:
statuses = pd.Series([res.route.route_status.value for _, res in results])
print(statuses.value_counts())
print(f"human-review ratio: {(statuses == 'HumanReview').mean():.0%}")

## Classification accuracy vs gold labels

In [ ]:
correct = sum(1 for r, res in results if res.classification and res.classification.category.value == r['gold_category'])
print(f'classification accuracy: {correct}/{len(results)} = {correct/len(results):.0%}')

## Latency / throughput snapshot

In [ ]:
print(f'total: {elapsed*1000:.1f} ms for {len(results)} complaints')
print(f'avg latency: {elapsed/len(results)*1000:.2f} ms/complaint')
print(f'throughput: {len(results)/elapsed:.0f} complaints/sec (mock backend)')